# Code for implementing the traditional Huff and the time-aware dynamic Huff model (T-Huff)

### Reference: Liang, Y., Gao, S., Cai, Y., Foutz, N. Z., & Wu, L. (2020). Calibrating the dynamic Huff model for business analysis using location big data. Transactions in GIS. 24(3), 1-23. DOI: 10.1111/tgis.12624. arXiv preprint available at https://arxiv.org/abs/2003.10857.

### Data: SafeGraph (https://shop.safegraph.com/)

<img src="https://geods.geography.wisc.edu/wp-content/uploads/2020/04/tgis12624_prediction.jpg"
     alt="Preiction vs. Actual"
     style="float: left; margin-right: 10px;" />

# import the pyswarms lib

In [1]:
import sys

In [2]:
# Import modules
import numpy as np

# Import PySwarms
#import pyswarms as ps
from pyswarms.utils.functions import single_obj as fx

# Some more magic so that the notebook will reload external python modules;
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
#%load_ext autoreload
#%autoreload 2

# define the Huff model

In [3]:
def HuffModel(mx,alpha,beta,att_field):
    #get the sum of H (dinominator)
    #print(alpha)
    #print(beta)
    cbg_sum = {}
    for i,row in mx.iterrows():
        if mx.loc[i]["census_block_group"] not in cbg_sum.keys():
            A = mx.loc[i][att_field]
            D = mx.loc[i]["distance"]
            cbg_sum[mx.loc[i]["census_block_group"]] = HuffPro(A,D,alpha,beta)
        else:
            A = mx.loc[i][att_field]
            D = mx.loc[i]["distance"]
            cbg_sum[mx.loc[i]["census_block_group"]] += HuffPro(A,D,alpha,beta)
    mx["pro"] = 0.0
    mx["h_sum"] = 0.0
    for i,row in mx.iterrows():
        mx.loc[i,"h_sum"] = cbg_sum[mx.loc[i]["census_block_group"]]
        A = mx.loc[i][att_field]
        D = mx.loc[i]["distance"]
        mx.loc[i,"pro"] =  HuffPro(A,D,alpha,beta)/mx.loc[i]["h_sum"]
    return mx
    

In [4]:
def HuffPro(A,D,alpha,beta):
    return pow(A,alpha)/pow(D,beta)

In [5]:
#"Whole Foods Market"
def Corr(mx,alpha,beta,att_field,location_name):
    mx_hf = HuffModel(mx,alpha,beta,att_field)
    mx_hf["pro"] = mx_hf["pro"].astype(float)
    mx_hf["actual_pro"] = mx_hf["actual_pro"].astype(float)
    mx_hf_location = mx_hf[mx_hf["location_name"] == location_name]
    cor = mx_hf_location[["actual_pro","pro"]].corr().values[0][1]
    #cor_total = mx_hf[["actual_pro","pro"]].corr().values[0][1]
    return mx_hf, cor

In [6]:
def THuffModelHourlyLocation(mx,alpha,beta,att_field,location_name,hourlyfield):
    cbg_sum = {}
    for i,row in mx.iterrows():
        if mx.loc[i]["census_block_group"] not in cbg_sum.keys():
            A = mx.loc[i][att_field]
            D = mx.loc[i]["distance"]
            cbg_sum[mx.loc[i]["census_block_group"]] = HuffPro(A,D,alpha,beta)
        else:
            A = mx.loc[i][att_field]
            D = mx.loc[i]["distance"]
            cbg_sum[mx.loc[i]["census_block_group"]] += HuffPro(A,D,alpha,beta)
    mx_location = mx[mx["location_name"] == location_name]
    mx_location["pro_total"] = 0.0
    mx_location["h_sum"] = 0.0
    mx_location["pro_hourly_est"] = None
    mx_location["pro_hourly_actual"] = None
    for i,row in mx_location.iterrows():
        mx_location.loc[i,"h_sum"] = cbg_sum[mx_location.loc[i]["census_block_group"]]
        A = mx_location.loc[i][att_field]
        D = mx_location.loc[i]["distance"]
        mx_location.loc[i,"pro_total"] =  HuffPro(A,D,alpha,beta)/mx_location.loc[i]["h_sum"]
        tem_dic_est = {}
        tem_dic_act = {}
        for key in mx_location.loc[i][hourlyfield].keys():
            tem_dic_est[key] = [x*mx_location.loc[i]["pro_total"] for x in mx_location.loc[i][hourlyfield][key]]
            tem_dic_act[key] = [x*mx_location.loc[i]["actual_pro"] for x in mx_location.loc[i][hourlyfield][key]]
        mx_location.at[i,"pro_hourly_est"] = tem_dic_est
        mx_location.at[i,"pro_hourly_actual"] = tem_dic_act
    return mx_location

In [7]:
def CorrHourlyLocation(mx,alpha,beta,att_field,location_name,hourlyfield,actual_field,est_field):
    mx_hf_location = THuffModelHourlyLocation(mx,alpha,beta,att_field,location_name,hourlyfield)
    #print(mx_hf)
    actual_list = []
    est_list = []
    for i,row in mx_hf_location.iterrows():
        actual_list.extend(mx_hf_location.loc[i][actual_field]['Monday'])
        actual_list.extend(mx_hf_location.loc[i][actual_field]['Tuesday'])
        actual_list.extend(mx_hf_location.loc[i][actual_field]['Wednesday'])
        actual_list.extend(mx_hf_location.loc[i][actual_field]['Thursday'])
        actual_list.extend(mx_hf_location.loc[i][actual_field]['Friday'])
        actual_list.extend(mx_hf_location.loc[i][actual_field]['Saturday'])
        actual_list.extend(mx_hf_location.loc[i][actual_field]['Sunday'])
        est_list.extend(mx_hf_location.loc[i][est_field]['Monday'])
        est_list.extend(mx_hf_location.loc[i][est_field]['Tuesday'])
        est_list.extend(mx_hf_location.loc[i][est_field]['Wednesday'])
        est_list.extend(mx_hf_location.loc[i][est_field]['Thursday'])
        est_list.extend(mx_hf_location.loc[i][est_field]['Friday'])
        est_list.extend(mx_hf_location.loc[i][est_field]['Saturday'])
        est_list.extend(mx_hf_location.loc[i][est_field]['Sunday'])
        #print(actual_list)
        #print(est_list)
        cor = numpy.corrcoef(actual_list,est_list)[0][1]
    return mx_hf_location, cor

# Read the data we processed. More raw data can be found from SafeGraph Inc. 

In [8]:
import pandas as pd
import numpy
infile = "../data/la_wholefoods.csv"
mx = pd.read_csv(infile,dtype={'census_block_group': str})
mx["raw_visit_counts"] = mx["raw_visit_counts"].astype(float)
mx["distance"] = mx["distance"].astype(float)
import ast
mx["popularity_day_pro"] = mx["popularity_day_pro"].apply(ast.literal_eval)
mx["popularity_hourly_pro"] = mx["popularity_hourly_pro"].apply(ast.literal_eval)

# Traditonal Huff model

## can input alpha, beta directly to get the correlation.

In [9]:
alpha_list_ori_detailed = [0.1,0.5,1,2,5]
beta_list = [0.1,0.5,1,2,5]
result = {}
for alpha in alpha_list_ori_detailed:
    for beta in beta_list:
        att_field = "raw_visit_counts"
        location_name = "Whole Foods Market" 
        mx_hf,cor = Corr(mx,alpha,beta,att_field,location_name)
        result[alpha,beta] = cor
        print(alpha, ",",beta,": ",cor)

0.1 , 0.1 :  0.08425880283525151
0.1 , 0.5 :  0.0848292383096415
0.1 , 1 :  0.08540094345880357
0.1 , 2 :  0.08609477374434608
0.1 , 5 :  0.08568810546570421
0.5 , 0.1 :  0.08431489378437948
0.5 , 0.5 :  0.08494972771518879
0.5 , 1 :  0.0856008779787165
0.5 , 2 :  0.08643392034373974
0.5 , 5 :  0.08614579224451198
1 , 0.1 :  0.08425426133554095
1 , 0.5 :  0.08496591043592747
1 , 1 :  0.08571705144722672
1 , 2 :  0.08674205971996579
1 , 5 :  0.08670241733904005
2 , 0.1 :  0.08374615268209332
2 , 0.5 :  0.08458006950681139
2 , 1 :  0.08550662089654612
2 , 2 :  0.08692301932052353
2 , 5 :  0.08769066455002994
5 , 0.1 :  0.08067898341130086
5 , 0.5 :  0.08153362512365303
5 , 1 :  0.08256709421051822
5 , 2 :  0.0844829259193276
5 , 5 :  0.08823245120857458


## optimization to find the alpha, beta with a range (0,2)

In [10]:
def PSOfunction_Huff(x): 
    print(x)
    import timeit
    #start_time = timeit.default_timer()
    alpha = x[:,0]
    beta = x[:,1]
    i = 0
    max_cor = -9999
    att_field = "raw_visit_counts"
    location_name = "Whole Foods Market"
    n_particles = x.shape[0]
    j = []
    for i in range(n_particles):
        mx_hf,cor = Corr(mx,alpha[i],beta[i],att_field,location_name)
        j.append(-cor)
    print(j)
    return np.array(j)

In [11]:
from pyswarms.single.global_best import GlobalBestPSO

#using Whole Foods with 
x_max = 2* np.ones(2)
x_min = 0* np.ones(2)
bounds = (x_min, x_max)
print(bounds)
options = {'c1': 2, 'c2': 2, 'w': 0.9}
#init_pos = pos
optimizer = GlobalBestPSO(n_particles=10, dimensions=2,options=options, bounds=bounds)

#kwargs={"mx": mx}
cost, pos = optimizer.optimize(PSOfunction_Huff, 10)

2026-10-21 09:24:40,071 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 2, 'c2': 2, 'w': 0.9}


(array([0., 0.]), array([2., 2.]))


pyswarms.single.global_best:   0%|          |0/10

[[0.21198982 0.37352583]
 [1.64728831 0.21775546]
 [0.00627794 1.92518414]
 [1.50968885 0.50483983]
 [0.9108442  0.44663728]
 [0.34041476 1.19418555]
 [1.51432185 1.12737843]
 [1.71832941 0.23400661]
 [0.9269752  1.65897395]
 [1.31910554 0.07835403]]


pyswarms.single.global_best:  10%|█         |1/10, best_cost=-0.0864

...

[np.float64(-0.08606206776072221), np.float64(-0.08644894985075138), np.float64(-0.08651858947646723), np.float64(-0.0863119376036009), np.float64(-0.08426283398454622), np.float64(-0.08496854444949106), np.float64(-0.08545052467565317), np.float64(-0.08549625356845846), np.float64(-0.08623307856919771), np.float64(-0.0855503923239256)]
[[1.09208245 1.34059154]
 [0.11832015 1.51927901]
 [0.91467808 1.64608228]
 [1.14861306 0.88801196]
 [0.69546486 1.95619681]
 [1.31222969 1.70574645]
 [0.78750002 0.76910992]
 [1.50872932 0.86262132]
 [0.46612022 1.10440473]
 [0.79138435 0.17578311]]


pyswarms.single.global_best:  90%|█████████ |9/10, best_cost=-0.0868

[np.float64(-0.08615521424392697), np.float64(-0.08584724219074878), np.float64(-0.08642048776438514), np.float64(-0.085561379446612), np.float64(-0.08654367009450445), np.float64(-0.08659072745112618), np.float64(-0.08537848899925983), np.float64(-0.08546362652087319), np.float64(-0.08570204631346252), np.float64(-0.08443314108875645)]
[[0.17497219 0.36783325]
 [1.38436988 1.81065275]
 [1.46737921 0.10588715]
 [1.1569228  0.75594853]
 [1.49503967 0.91862843]
 [0.93478188 1.97927421]
 [0.56043822 0.76486461]
 [0.89923058 1.92526443]
 [0.94528714 1.70776838]
 [0.90400306 0.02926598]]


pyswarms.single.global_best: 100%|██████████|10/10, best_cost=-0.0868
2026-10-21 09:24:42,787 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.08679715225686313, best pos: [1.15460576 1.98474071]


[np.float64(-0.08467775687534043), np.float64(-0.08670824025905668), np.float64(-0.0840873352617041), np.float64(-0.08536033649129067), np.float64(-0.08555713546034767), np.float64(-0.08669534251170928), np.float64(-0.08533084240536055), np.float64(-0.08663902421338074), np.float64(-0.08648709713772319), np.float64(-0.08414354999256209)]


In [12]:
cost,pos

(np.float64(-0.08679715225686313), array([1.15460576, 1.98474071]))

# T-Huff model

## considering the temporal visit changes
<img src="https://geods.geography.wisc.edu/wp-content/uploads/2020/04/tgis12624_HourlyPlot.png"
     alt="Preiction vs. Actual"
     style="float: left; margin-right: 10px;" />

## can input alpha, beta directly to get the correlation.

In [13]:
alpha_list_ori_detailed = [0.1,0.5,1,2,5]
beta_list = [0.1,0.5,1,2,5]
result_dic_ori_detailed_hourly = {}
for alpha in alpha_list_ori_detailed:
    for beta in beta_list:
        att_field = "raw_visit_counts"
        location_name = "Whole Foods Market" 
        mx_hf,cor = CorrHourlyLocation(mx,alpha,beta,att_field,location_name,"popularity_hourly_pro",'pro_hourly_actual','pro_hourly_est')
        result_dic_ori_detailed_hourly[alpha,beta] = cor
        print(alpha, ",",beta,": ",cor)

0.1 , 0.1 :  0.4405067092660758
0.1 , 0.5 :  0.44064026136496687
0.1 , 1 :  0.4407645558497947
0.1 , 2 :  0.4408774050690581
0.1 , 5 :  0.4403958257319721
0.5 , 0.1 :  0.4405066420641679
0.5 , 0.5 :  0.4406592112849741
0.5 , 1 :  0.44080693344781297
0.5 , 2 :  0.44096202757068603
0.5 , 5 :  0.44053355954129875
1 , 0.1 :  0.44046739019944164
1 , 0.5 :  0.44064292759134616
1 , 1 :  0.4408201506033011
1 , 2 :  0.44103234216068116
1 , 5 :  0.44069749504447703
2 , 0.1 :  0.4402706553638804
2 , 0.5 :  0.44048477457055246
2 , 1 :  0.440715355848076
2 , 2 :  0.4410435751354211
2 , 5 :  0.44097624105915023
5 , 0.1 :  0.43913903637876417
5 , 0.5 :  0.4393760110181551
5 , 1 :  0.4396574991675437
5 , 2 :  0.4401631559216132
5 , 5 :  0.4410307723926893


## can also just use optimation to get optimal alpha, beta

In [14]:
def PSOfunction_THuff(x): 
    print(x)
    import timeit
    #start_time = timeit.default_timer()
    alpha = x[:,0]
    beta = x[:,1]
    i = 0
    max_cor = -9999
    att_field = "raw_visit_counts"
    location_name = "Whole Foods Market"
    n_particles = x.shape[0]
    j = []
    for i in range(n_particles):
        mx_hf,cor = CorrHourlyLocation(mx,alpha[i],beta[i],att_field,location_name,"popularity_hourly_pro",'pro_hourly_actual','pro_hourly_est')
        j.append(-cor)
    print(j)
    return np.array(j)

In [15]:
from pyswarms.single.global_best import GlobalBestPSO

#using Whole Foods with 
x_max = 2* np.ones(2)
x_min = 0* np.ones(2)
bounds = (x_min, x_max)
print(bounds)
options = {'c1': 2, 'c2': 2, 'w': 0.9}
#init_pos = pos
optimizer = GlobalBestPSO(n_particles=10, dimensions=2,options=options, bounds=bounds)

#kwargs={"mx": mx}
cost, pos = optimizer.optimize(PSOfunction_THuff, 10)

2026-10-21 09:25:03,208 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 2, 'c2': 2, 'w': 0.9}


(array([0., 0.]), array([2., 2.]))


pyswarms.single.global_best:   0%|          |0/10

[[0.88504925 0.28745472]
 [1.95732843 0.15206198]
 [1.97330791 0.3894682 ]
 [1.46989811 1.13364558]
 [0.00865707 1.7474257 ]
 [0.24924073 1.89537382]
 [0.34550689 0.46470446]
 [1.01048432 1.1177956 ]
 [1.05369718 0.6790757 ]
 [0.73464531 1.97533299]]


pyswarms.single.global_best:  10%|█         |1/10, best_cost=-0.441

...

[np.float64(-0.4405283272168629), np.float64(-0.44082415483699394), np.float64(-0.4405005557707475), np.float64(-0.44096788481694993), np.float64(-0.4410060056947418), np.float64(-0.44085190050737405), np.float64(-0.44085158795676105), np.float64(-0.4410266958566341), np.float64(-0.44077883487654373), np.float64(-0.440990299256154)]
[[1.0713084  1.57030179]
 [0.29118167 0.49826708]
 [1.20625488 0.77741825]
 [1.14801748 1.61747977]
 [1.06271766 0.46395946]
 [0.51831483 1.63110635]
 [1.05595422 1.42376926]
 [1.03575375 0.6804077 ]
 [1.74212266 1.29071019]
 [1.2122737  1.47776158]]


pyswarms.single.global_best:  90%|█████████ |9/10, best_cost=-0.441

[np.float64(-0.4409672999030451), np.float64(-0.4406523234005724), np.float64(-0.44073423758131386), np.float64(-0.4409795745571678), np.float64(-0.4406229456427121), np.float64(-0.44092858225842296), np.float64(-0.44093462368477965), np.float64(-0.44071023314107893), np.float64(-0.4408645699994132), np.float64(-0.4409491982107492)]
[[1.07469079 0.23325435]
 [1.76977354 0.42477012]
 [1.56072645 0.73325871]
 [1.37225015 1.38173707]
 [1.11424529 0.97373851]
 [0.11195736 1.62397846]
 [0.28165425 1.48482096]
 [1.13674808 1.09752771]
 [1.5931334  1.75500297]
 [0.76578056 1.25044559]]


pyswarms.single.global_best: 100%|██████████|10/10, best_cost=-0.441
2026-10-21 09:26:23,518 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.44103820717369063, best pos: [1.08842231 1.98305878]


[np.float64(-0.44052075192714263), np.float64(-0.4404981714235264), np.float64(-0.4406766346568597), np.float64(-0.44092115967207446), np.float64(-0.44080843582579327), np.float64(-0.4408577669132965), np.float64(-0.44087334226787567), np.float64(-0.44084660785202434), np.float64(-0.4410086491273683), np.float64(-0.44088344867861035)]


In [16]:
cost, pos

(np.float64(-0.44103820717369063), array([1.08842231, 1.98305878]))

## output the data for mapping in Huff model

In [17]:
att_field = "raw_visit_counts"
location_name = "Whole Foods Market"
pos = [0.717,0.805]
alpha = pos[0]
beta = pos[1]
mx_hf,cor = Corr(mx,alpha,beta,att_field,location_name)

In [18]:
cor

np.float64(0.0854177921445047)

In [19]:
mx_hf_wholefoods = mx_hf[mx_hf['location_name'] == "Whole Foods Market"]

In [20]:
list1 = mx_hf_wholefoods['actual_pro']
list2 = mx_hf_wholefoods['pro']

In [21]:
numpy.corrcoef(list1,list2)

array([[1.        , 0.08541779],
       [0.08541779, 1.        ]])

## The correlation result for the estimated hourly visit probability using the T-Huff model compared to the actual visit probability observed in SafeGraph POI visits.

In [22]:
att_field = "raw_visit_counts"
location_name = "Whole Foods Market"
pos = [0.787,0.765]
alpha = pos[0]
beta = pos[1]
mx_hf,cor = CorrHourlyLocation(mx,alpha,beta,att_field,location_name,"popularity_hourly_pro",'pro_hourly_actual','pro_hourly_est')

In [23]:
cor

np.float64(0.44074849924184933)